# ASVspoof5 shard download check

Run this in a Google Colab runtime with a mounted Drive. The goal is to confirm whether `flac_D_aa.tar` downloads successfully with Xet disabled.

In [ ]:
import os
import tarfile
from pathlib import Path

from huggingface_hub import hf_hub_download

os.environ["HF_HUB_DISABLE_XET"] = "1"

base = Path('/content/asvspoof5')
base.mkdir(parents=True, exist_ok=True)
protocol_tar = base / 'ASVspoof5_protocols.tar'

if not protocol_tar.exists():
    print('Downloading protocol archive to temporary Colab storage...')
    protocol_tar = Path(hf_hub_download(
        repo_id='jungjee/asvspoof5',
        repo_type='dataset',
        filename='ASVspoof5_protocols.tar',
        local_dir=str(base),
    ))
else:
    print('Protocol archive already exists:', protocol_tar)

protocol_path = base / 'ASVspoof5.dev.track_1.tsv'
if not protocol_path.exists():
    print('Extracting protocol files into temporary Colab storage...')
    with tarfile.open(protocol_tar, 'r') as tf:
        tf.extractall(base)
else:
    print('Protocol files already extracted:', protocol_path)

print('PROTOCOL_PATH=', protocol_path)
print('PROTOCOL_READY=', protocol_path.exists())

ASVspoof5_protocols.tar: reconstructing file:   0%|          |  0.00B / 93.0MB            

ASVspoof5_protocols.tar: downloading bytes:           |  0.00B            

Extracting protocol files into temporary Colab storage...
PROTOCOL_PATH= /content/asvspoof5/ASVspoof5.dev.track_1.tsv
PROTOCOL_READY= True


/tmp/ipykernel_612/1449341935.py:28: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tf.extractall(base)


In [ ]:
import csv
import os
import shutil
import tarfile
from pathlib import Path

from huggingface_hub import hf_hub_download

os.environ["HF_HUB_DISABLE_XET"] = "1"

storage_base = Path('/content/drive/MyDrive/ElderTrustShield/data/audio/asvspoof5')
storage_base.mkdir(parents=True, exist_ok=True)
protocol_base = Path('/content/asvspoof5')
extract_dir = Path('/content/flac_D_aa')
local_tar = Path('/content/flac_D_aa.tar')
drive_archive_path = storage_base / 'flac_D_aa.tar'
protocol_path = protocol_base / 'ASVspoof5.dev.track_1.tsv'
expected_audio_count = 47400

if not drive_archive_path.exists():
    print('Downloading audio shard to Google Drive...')
    drive_archive_path = Path(hf_hub_download(
        repo_id='jungjee/asvspoof5', repo_type='dataset',
        filename='flac_D_aa.tar', local_dir=str(storage_base),
    ))
else:
    print('Audio archive already exists on Drive:', drive_archive_path)

if not protocol_path.exists():
    raise FileNotFoundError(f'Missing protocol file: {protocol_path}. Run Cell 1 first.')

# Copy to local disk first — reading the tar over Drive's network mount during
# extraction of tens of thousands of members is the likely disconnect cause.
# A single sequential copy is far more resilient than 47,400 small FUSE reads.
if not local_tar.exists() or local_tar.stat().st_size != drive_archive_path.stat().st_size:
    print('Copying archive to local disk...')
    shutil.copy(drive_archive_path, local_tar)
else:
    print('Local copy of archive already present.')

extract_dir.mkdir(parents=True, exist_ok=True)

with tarfile.open(local_tar, 'r') as tf:
    members = tf.getmembers()
    total = len(members)
    print(f'Archive contains {total} members. Extracting (resumable)...')
    for i, member in enumerate(members):
        target = extract_dir / member.name
        if not target.exists():
            tf.extract(member, extract_dir, filter='data')
        if i % 2000 == 0:
            print(f'{i}/{total} processed', flush=True)

final_audio_count = sum(1 for _ in extract_dir.rglob('*.flac'))
print('FINAL_FLAC_COUNT=', final_audio_count)
if final_audio_count != expected_audio_count:
    raise ValueError(
        f'Extraction incomplete: expected {expected_audio_count}, found {final_audio_count}. '
        'Re-run this cell — it will skip already-extracted files and resume.'
    )

# Confirmed schema (cross-checked against published ASVspoof5 protocol description):
# speaker_id filename gender [reserved] attack_configuration attack_algorithm label placeholder
present = {p.stem: p.relative_to(extract_dir).as_posix() for p in sorted(extract_dir.rglob('*.flac'))}
rows = []
with protocol_path.open('r', encoding='utf-8') as fh:
    for line in fh:
        parts = line.strip().split()
        if len(parts) < 10:
            continue
        filename_key, label, attack_algorithm = parts[1], parts[-2], parts[-3]
        if label in {'bonafide', 'spoof'} and filename_key in present:
            rows.append((present[filename_key], label, attack_algorithm))

out_csv = storage_base / 'flac_D_aa_labels.csv'
with out_csv.open('w', newline='', encoding='utf-8') as fh:
    writer = csv.writer(fh)
    writer.writerow(['filename', 'label', 'attack_algorithm'])
    writer.writerows(rows)

counts = {'bonafide': 0, 'spoof': 0}
for _, label, _ in rows:
    counts[label] += 1

print('TOTAL_MATCHED_ROWS=', len(rows))
print('REAL_BONAFIDE=', counts['bonafide'])
print('FAKE_SPOOF=', counts['spoof'])
print('CSV_PATH=', out_csv)

flac_D_aa.tar: reconstructing file:   0%|          |  0.00B / 6.65GB            

flac_D_aa.tar: downloading bytes:           |  0.00B            

Copying archive to local disk...
Archive contains 47400 members. Extracting (resumable)...
0/47400 processed
2000/47400 processed
4000/47400 processed
6000/47400 processed
8000/47400 processed
10000/47400 processed
12000/47400 processed
14000/47400 processed
16000/47400 processed
18000/47400 processed
20000/47400 processed
22000/47400 processed
24000/47400 processed
26000/47400 processed
28000/47400 processed
30000/47400 processed
32000/47400 processed
34000/47400 processed
36000/47400 processed
38000/47400 processed
40000/47400 processed
42000/47400 processed
44000/47400 processed
46000/47400 processed
FINAL_FLAC_COUNT= 47400
TOTAL_MATCHED_ROWS= 47400
REAL_BONAFIDE= 10700
FAKE_SPOOF= 36700
CSV_PATH= /content/drive/MyDrive/ElderTrustShield/data/audio/asvspoof5/flac_D_aa_labels.csv


In [ ]:
import importlib.metadata
import os
import subprocess
import sys
from pathlib import Path

os.environ["HF_HUB_DISABLE_XET"] = "1"

required_versions = {
    "transformers": "4.52.4",
    "huggingface-hub": "0.32.4",
    "tokenizers": "0.21.1",
}

installed_versions = {}
for package_name in required_versions:
    try:
        installed_versions[package_name] = importlib.metadata.version(package_name)
    except importlib.metadata.PackageNotFoundError:
        installed_versions[package_name] = None

packages_need_install = [
    f"{package_name}=={version}"
    for package_name, version in required_versions.items()
    if installed_versions[package_name] != version
 ]

if packages_need_install:
    print("Installing compatible Transformers packages:", packages_need_install)
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--force-reinstall",
            "--no-cache-dir",
            "--no-deps",
            *packages_need_install,
        ],
        check=True,
    )
    print()
    print("Installation finished.")
    print("Restart the Colab runtime manually, reconnect, and run this cell again.")
    raise RuntimeError(
        "Runtime restart required after package installation."
    )

repo_dir = Path("/content/AASIST3")
if not repo_dir.exists():
    subprocess.run(
        ["git", "clone", "https://github.com/mtuciru/AASIST3.git", str(repo_dir)],
        check=True,
    )

sys.path.insert(0, str(repo_dir))

import torch
import torchaudio
from model import aasist3

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("DEVICE=", device)
if device.type != "cuda":
    raise RuntimeError(
        "A GPU runtime is required. In Colab choose Runtime > Change runtime type > T4 GPU, "
        "restart the runtime, and rerun this cell."
    )

model = aasist3.from_pretrained("MTUCI/AASIST3")
model = model.to(device)
model.eval()

print("MODEL_LOADED=", type(model).__name__)

DEVICE= cuda


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/550 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/1.29G [00:00<?, ?B/s]

MODEL_LOADED= aasist3


In [ ]:
import csv
import pickle
from collections import defaultdict
from pathlib import Path

import numpy as np
from sklearn.metrics import accuracy_score, confusion_matrix, roc_curve
from tqdm.auto import tqdm

csv_path = Path('/content/drive/MyDrive/ElderTrustShield/data/audio/asvspoof5/flac_D_aa_labels.csv')
audio_base = Path('/content/flac_D_aa')
checkpoint_path = Path('/content/drive/MyDrive/ElderTrustShield/data/audio/asvspoof5/inference_checkpoint.pkl')
target_sample_rate = 16000
target_samples = 64600
batch_size = 16
checkpoint_every = 50  # save roughly every 800 files

with csv_path.open('r', encoding='utf-8', newline='') as fh:
    rows = list(csv.DictReader(fh))

print('CSV_ROWS=', len(rows))
if len(rows) != 47400:
    raise ValueError(f'Expected 47,400 rows, found {len(rows)}.')

label_to_int = {'bonafide': 0, 'spoof': 1}
resampler_cache = {}

def load_and_preprocess(audio_path):
    waveform, sample_rate = torchaudio.load(str(audio_base / audio_path))
    if sample_rate != target_sample_rate:
        if sample_rate not in resampler_cache:
            resampler_cache[sample_rate] = torchaudio.transforms.Resample(sample_rate, target_sample_rate)
        waveform = resampler_cache[sample_rate](waveform)
    if waveform.shape[0] > 1:
        waveform = waveform.mean(dim=0, keepdim=True)
    waveform = waveform.squeeze(0)
    if waveform.shape[0] < target_samples:
        waveform = torch.nn.functional.pad(waveform, (0, target_samples - waveform.shape[0]))
    else:
        waveform = waveform[:target_samples]
    return waveform

# Resume from a checkpoint on Drive if one exists (survives even a full runtime restart, not just a websocket blip)
if checkpoint_path.exists():
    with checkpoint_path.open('rb') as f:
        state = pickle.load(f)
    true_labels, predicted_labels = state['true_labels'], state['predicted_labels']
    bonafide_scores, attack_algorithms = state['bonafide_scores'], state['attack_algorithms']
    start_row = state['next_row']
    print(f'Resuming: {len(true_labels)} files already done, continuing from row {start_row}')
else:
    true_labels, predicted_labels, bonafide_scores, attack_algorithms = [], [], [], []
    start_row = 0

with torch.inference_mode():
    starts = list(range(start_row, len(rows), batch_size))
    for batch_num, start in enumerate(tqdm(starts, desc='Running AASIST3 inference')):
        batch_rows = rows[start:start + batch_size]
        batch_waveforms = torch.stack(
            [load_and_preprocess(row['filename']) for row in batch_rows]
        ).to(device, non_blocking=True)

        logits = model(batch_waveforms)
        probabilities = torch.softmax(logits, dim=1)
        predictions = torch.argmax(probabilities, dim=1)

        true_labels.extend(label_to_int[row['label'].strip().lower()] for row in batch_rows)
        predicted_labels.extend(predictions.cpu().tolist())
        bonafide_scores.extend(probabilities[:, 0].cpu().tolist())
        attack_algorithms.extend(row.get('attack_algorithm', row.get('attack_type_code', 'unknown')) for row in batch_rows)

        if (batch_num + 1) % checkpoint_every == 0:
            with checkpoint_path.open('wb') as f:
                pickle.dump({
                    'true_labels': true_labels, 'predicted_labels': predicted_labels,
                    'bonafide_scores': bonafide_scores, 'attack_algorithms': attack_algorithms,
                    'next_row': start + batch_size,
                }, f)

# final checkpoint at completion
with checkpoint_path.open('wb') as f:
    pickle.dump({
        'true_labels': true_labels, 'predicted_labels': predicted_labels,
        'bonafide_scores': bonafide_scores, 'attack_algorithms': attack_algorithms,
        'next_row': len(rows),
    }, f)

CSV_ROWS= 47400


Running AASIST3 inference:   0%|          | 0/2963 [00:00<?, ?it/s]

In [ ]:
true_labels = np.asarray(true_labels)
predicted_labels = np.asarray(predicted_labels)
bonafide_scores = np.asarray(bonafide_scores)
attack_algorithms = np.asarray(attack_algorithms)

accuracy = accuracy_score(true_labels, predicted_labels)
matrix = confusion_matrix(true_labels, predicted_labels, labels=[0, 1])

bona_fide_targets = (true_labels == 0).astype(np.int32)
fpr, tpr, _ = roc_curve(bona_fide_targets, bonafide_scores)
eer_index = np.nanargmin(np.abs(fpr - (1.0 - tpr)))
eer = (fpr[eer_index] + (1.0 - tpr[eer_index])) / 2.0

print('OVERALL_RESULTS')
print('FILES_EVALUATED=', len(true_labels))
print(f'ACCURACY={accuracy:.6f}')
print(f'EER={eer:.6f} ({eer * 100:.2f}%)')
print()
print('CONFUSION_MATRIX (rows: true [bonafide, spoof], cols: predicted [bonafide, spoof])')
print(matrix)
print()
print('BREAKDOWN_BY_ATTACK_ALGORITHM')
groups = defaultdict(list)
for index, aa in enumerate(attack_algorithms):
    groups[aa].append(index)
print(f"{'attack_algorithm':<20}{'files':>8}{'accuracy':>14}{'bonafide':>12}{'spoof':>10}")
for aa in sorted(groups):
    idx = np.asarray(groups[aa])
    group_acc = accuracy_score(true_labels[idx], predicted_labels[idx])
    print(f'{aa:<20}{len(idx):>8}{group_acc:>14.6f}{int(np.sum(true_labels[idx]==0)):>12}{int(np.sum(true_labels[idx]==1)):>10}')

OVERALL_RESULTS
FILES_EVALUATED= 47400
ACCURACY=0.181308
EER=0.807705 (80.77%)

CONFUSION_MATRIX (rows: true [bonafide, spoof], cols: predicted [bonafide, spoof])
[[ 8115  2585]
 [36221   479]]

BREAKDOWN_BY_ATTACK_ALGORITHM
attack_algorithm       files      accuracy    bonafide     spoof
A09                     4577      0.000000           0      4577
A10                     4552      0.000000           0      4552
A11                     4599      0.000000           0      4599
A12                     4692      0.080776           0      4692
A13                     4509      0.000000           0      4509
A14                     4616      0.009965           0      4616
A15                     4491      0.000000           0      4491
A16                     4664      0.011578           0      4664
bonafide               10700      0.758411       10700         0


In [ ]:
import pickle
import numpy as np
from pathlib import Path
from sklearn.metrics import accuracy_score, confusion_matrix, roc_curve
from collections import defaultdict

checkpoint_path = Path('/content/drive/MyDrive/ElderTrustShield/data/audio/asvspoof5/inference_checkpoint.pkl')
with checkpoint_path.open('rb') as f:
    state = pickle.load(f)

true_labels = np.asarray(state['true_labels'])
attack_algorithms = np.asarray(state['attack_algorithms'])

# Swap: treat the model's raw index 1 as bonafide-probability, index 0 as spoof
predicted_labels = 1 - np.asarray(state['predicted_labels'])
bonafide_scores = 1.0 - np.asarray(state['bonafide_scores'])

accuracy = accuracy_score(true_labels, predicted_labels)
matrix = confusion_matrix(true_labels, predicted_labels, labels=[0, 1])
bona_fide_targets = (true_labels == 0).astype(np.int32)
fpr, tpr, _ = roc_curve(bona_fide_targets, bonafide_scores)
eer_index = np.nanargmin(np.abs(fpr - (1.0 - tpr)))
eer = (fpr[eer_index] + (1.0 - tpr[eer_index])) / 2.0

print(f'ACCURACY={accuracy:.6f}')
print(f'EER={eer:.6f} ({eer*100:.2f}%)')
print(matrix)

groups = defaultdict(list)
for i, aa in enumerate(attack_algorithms):
    groups[aa].append(i)
for aa in sorted(groups):
    idx = np.asarray(groups[aa])
    print(aa, accuracy_score(true_labels[idx], predicted_labels[idx]))

ACCURACY=0.818692
EER=0.192295 (19.23%)
[[ 2585  8115]
 [  479 36221]]
A09 1.0
A10 1.0
A11 1.0
A12 0.9192242114236999
A13 1.0
A14 0.9900346620450606
A15 1.0
A16 0.9884219554030875
bonafide 0.24158878504672898


In [ ]:
threshold = fpr[eer_index]  # or: bonafide_scores sorted at the EER crossing point
# more directly, use the actual crossing threshold from roc_curve's third return value:
fpr, tpr, thresholds = roc_curve(bona_fide_targets, bonafide_scores)
eer_index = np.nanargmin(np.abs(fpr - (1.0 - tpr)))
eer_threshold = thresholds[eer_index]

calibrated_predictions = (bonafide_scores >= eer_threshold).astype(int)
calibrated_predictions = 1 - calibrated_predictions  # convert to true_labels' convention (0=bonafide,1=spoof)

calibrated_accuracy = accuracy_score(true_labels, calibrated_predictions)
calibrated_matrix = confusion_matrix(true_labels, calibrated_predictions, labels=[0, 1])
print(f'CALIBRATED_ACCURACY={calibrated_accuracy:.6f}')
print(calibrated_matrix)

CALIBRATED_ACCURACY=0.807932
[[ 8638  2062]
 [ 7042 29658]]
